# 08. Hands-on: Laya — a multilingual System 1 decision engine

[Laya](https://github.com/NandhaKishorM/laya) is an open-source (Apache-2.0) decision-model family released by Convai Innovations.

- **Non-autoregressive**: BERT-class encoders (English 421M / multilingual 322M) answer choice/noul/score in a single forward pass — the smallest decision models in this workshop (practical even on CPU).
- **100+ languages**: `Router()` detects the input language and automatically routes non-English input (such as Korean) to the `laya-multilingual` checkpoint.
- Returns **calibrated probabilities**, trained with RLCD (RL against strictly proper scoring rules).
- `laya[serve]` also speaks **`/v1/systemone`** (the Jev-compatible wire format), with LangChain/LangGraph/LlamaIndex/CrewAI/MCP integrations and a TypeScript port (`npm install laya-ts`).
- Fully free self-hosting — no API key, no account.

This notebook uses the **native Python `Router` API directly inside the kernel**, with no HTTP server (the kernel env needs `pip install laya` — preinstalled in the workshop kernel; checkpoints download into the HF_HOME cache).


In [1]:
import sys, os, time
os.environ["DMWS_LANG"] = "en"   # localize helper output/graph captions (set BEFORE importing ws_common)
from pathlib import Path; sys.path.insert(0, str(Path.cwd().parent))  # ws_common path
os.environ.setdefault("HF_HOME", os.environ.get("HF_HOME", str(Path.home() / ".cache/huggingface")))
import ws_common as W
from laya import Router

router = Router()    # downloads a checkpoint on first use
print("laya Router ready")


laya Router ready


## 1. Three decision types from one customer message

Laya lets you bundle several questions over one state. In a banking-support context we ask **routing (choice) + urgency (noul) + customer emotion (score)** at once over "My wallet was stolen! Please freeze my card right now!!". With English input, check that `routing.model` reports `english` — non-English input (for example the Korean utterances in the [KR] notebook) is auto-routed to the `laya-multilingual` checkpoint.


In [2]:
state = "My wallet was stolen! Please freeze my card right now!!"
questions = {
    "d2_team": {"type": "choice",
                "instructions": "Which banking support specialist team should handle this customer message?",
                "criteria": {c: None for c in
                             ["cards", "card_security", "disputes_fees", "transfers",
                              "accounts_balance", "top_up", "exchange_fx", "refunds",
                              "declined_pay", "identity_verif", "account_mgmt", "general_faq"]}},
    "urgent": {"type": "noul", "instructions": "Does this message convey urgency?"},
    "emotion": {"type": "score", "instructions": "How upset is the customer?",
                "criteria": ["calm", "worried", "very upset"]},
}

router.predict(state, questions)   # warm-up (excluded from timing)
t0 = time.perf_counter()
r = router.predict(state, questions)
ms = (time.perf_counter() - t0) * 1000

print(f"checkpoint: {r['routing']['model']}   (3 questions combined: {ms:,.1f}ms)")
a = r["answers"]
print(f"  d2_team -> {a['d2_team']['choice']} (confidence {a['d2_team']['confidence']:.3f})")
print(f"  urgent  -> P(yes) = {a['urgent']['noul']:.3f}")
print(f"  emotion -> score {a['emotion']['score']:.2f}  (0=calm ~ 2=very upset)")

/data/venvs/agent/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 2368.86it/s]

/data/venvs/agent/lib/python3.12/site-packages/laya/router.py:999: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(repo, **kwargs)


checkpoint: english   (3 questions combined: 28.9ms)
  d2_team -> card_security (confidence 0.999)
  urgent  -> P(yes) = 0.833
  emotion -> score 1.78  (0=calm ~ 2=very upset)


## 2. Plugging it into our LangGraph workflow — a 20-line adapter

Laya is a Python object API, so a **thin adapter** matching the `.decide()` signature drops it straight into the same graph — a demonstration of how small a backend swap is.
> The adapter **pins `model="multilingual"`**. If you leave routing automatic, long states mixed with English (tool JSON etc.) go to the english checkpoint and the repeated checkpoint switching slows everything down; long inputs also need `max_len=8192`. Laya also warns that **choice confidences with 11+ options are uncalibrated** (our D2 has 12) — use the label, avoid confidence-based designs there.


In [3]:
from agent.decisions import DecisionResult, _fallback

class LayaDecision:
    name = "laya"
    def __init__(self, router):
        self.router = router
    def decide(self, *, state_text, question, choices, decision_id):
        qs = {decision_id: {"type": "choice", "instructions": question,
                            "criteria": {c: None for c in choices}}}
        t0 = time.perf_counter()
        r = self.router.predict(state_text, qs, model="multilingual", max_len=8192)
        ms = (time.perf_counter() - t0) * 1000
        ans = r["answers"][decision_id]
        label, valid = ans["choice"], ans["choice"] in choices
        if not valid:
            label = _fallback(decision_id, choices)
        return DecisionResult(label=label, confidence=ans.get("confidence"),
                              latency_ms=round(ms, 2), valid=valid,
                              usage={}, cost_usd=0.0, raw=str(ans)[:200])

laya_dec = LayaDecision(router)
print("adapter ready")

adapter ready


In [4]:
from agent.graph_v2 import build_graph_v2
from agent.decisions import SGLangDecision

client, llm = W.make_clients()
graph_laya   = build_graph_v2(laya_dec, llm)
graph_sglang = build_graph_v2(SGLangDecision(port=8001), llm)
sc = W.scenario(1)
print("Scenario:", sc["title"], "|", sc["question"])

Scenario: Vague inquiry -> clarification loop | My card doesn't swipe. What should I do?


In [5]:
res_laya = W.run_verbose(graph_laya, sc["question"], sc["script"], "Laya")


[Laya] customer: My card doesn't swipe. What should I do?


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 2144.99it/s]


STEP 1 ▶ guardrail_check — D1 intake screening: in-scope / out-of-scope / compliance
      decision[D1] -> sensitive_compliance   (2,175ms, confidence 0.178)



STEP 2 ▶ respond — Sonnet writes the final customer reply
      generate (Sonnet 4.5)   (4,352ms)
      --- final reply ---
      Thank you for contacting us. I understand your card isn't swiping properly.
      
      For security reasons, I'm unable to process card replacement requests through this channel. However, I can help you get this resolved quickly:
      
      Please contact our secure customer service line at the number on the back of your card, or visit your nearest branch where our team can verify your identity and issue a replacement card immediately.
      
      In the meantime, you may be able to use contactless payment or insert your chip if those features are working.
      
      Is there anything else I can help you with today?

>>> [Laya] 1 decisions, total 2,175ms | generation 4,352ms | e2e 6,531ms


In [6]:
res_sglang = W.run_verbose(graph_sglang, sc["question"], sc["script"], "SGLang")


[SGLang] customer: My card doesn't swipe. What should I do?

STEP 1 ▶ guardrail_check — D1 intake screening: in-scope / out-of-scope / compliance
      decision[D1] -> in_scope   (53ms, confidence 1.000)

STEP 2 ▶ intent_router — D2 routing: pick one of 12 specialist teams
      decision[D2] -> declined_pay   (33ms, confidence 0.998)

STEP 3 ▶ clarify_gate — D5 sufficiency check: can we act now, or must we ask back?
      decision[D5] -> need_clarification   (32ms, confidence 1.000)



STEP 4 ▶ ask_clarify — Clarify: Sonnet writes a question -> customer reply received
      generate (Sonnet 4.5)   (1,626ms)
      support agent: Which card are you having trouble with - your debit card or credit card?
      customer: Since yesterday the card won't go through at in-store terminals only. Details: Visa debit card ending 7712, last attempt on 2026-10-27 at TechMart Gangnam, amount USD 34.60.

STEP 5 ▶ clarify_gate — D5 sufficiency check: can we act now, or must we ask back?
      decision[D5] -> info_sufficient   (35ms, confidence 1.000)

STEP 6 ▶ tool_step — D3 tool choice -> run tool -> D6 result review (done/retry/give up)
      decision[D3] -> explain_decline   (33ms, confidence 1.000)
      tool run: explain_decline   (0.0ms)
      decision[D6] -> goal_met   (34ms, confidence 0.994)



STEP 7 ▶ summarize — Sonnet condenses the tool result into customer-facing facts
      generate (Sonnet 4.5)   (3,737ms)
      summary (first line): Based on the tool result, here are the key facts:

STEP 8 ▶ risk_check — D7 risk check: fraud/AML monitoring flag present?
      decision[D7] -> no_risk   (39ms, confidence 1.000)

STEP 9 ▶ escalation_gate — D4 final action: resolve / clarify / human handoff
      decision[D4] -> clarify   (36ms, confidence 1.000)



STEP 10 ▶ respond — Sonnet writes the final customer reply
      generate (Sonnet 4.5)   (5,315ms)
      --- final reply ---
      Thank you for providing those details. I've looked into your Visa debit card ending in 7712 and the declined transaction at TechMart Gangnam on October 27th.
      
      To help resolve this issue, I need a bit more information:
      
      **Could you confirm:**
      - Have you successfully used this card for other transactions (online or ATM withdrawals) since yesterday?
      - Is your card physically damaged in any way (scratches on the chip or magnetic stripe)?
      - Did you recently travel internationally or change your contact information?
      
      This will help me identify whether it's a card issue, terminal compatibility problem, or account restriction.
      
      Reference ID: REF-ws-SGLang-162250

>>> [SGLang] 8 decisions, total 296ms | generation 10,677ms | e2e 10,981ms


In [7]:
W.compare(res_laya, res_sglang)

| metric | Laya | SGLang | ratio(Laya/SGLang) |
|---|---:|---:|---:|
| decisions | 1 | 8 | |
| **decision latency total** | **2,175ms** | **296ms** | **7.4x** |
| generation (LLM) latency | 4,352ms | 10,677ms | **0.4x** |
| e2e | 6,531ms | 10,981ms | **0.6x** |

**decision-layer speedup: 7.4x** (e2e saved -4,450ms)

**per-decision latency**

| node | decision | Laya | SGLang | ratio | label |
|---|---|---:|---:|---:|---|
| guardrail_check | D1 | 2,175ms | 53ms | **40.9x** | Laya=sensitive_compliance / SGLang=in_scope |

## Measured decision quality on the Korean bench (3-backend comparison)

D1/D2 measured over the workshop's 160-conversation Korean bench (the [KR] track inputs) with objective gold labels (the source project's decider-quality bench, the bench summary (source project)):

| backend | D1 guardrail (3-way) | D2 routing strict/lenient (12-way) | p50 latency |
|---|---|---|---|
| SGLang Qwen3-VL-8B | **0.913** | **0.629 / 0.707** | 33ms |
| strands-decider 2B | 0.613 | 0.500 / 0.593 | 85ms |
| Laya multilingual 322M | 0.419 | 0.357 / 0.379 | 21ms |

**Reading**: model capacity dominated the quality ceiling (8B > 2B > 0.3B), beating language specialization (Laya multilingual). The Laya number matches its own documented zero-shot score (0.362); Laya is a **"fine-tune me" model** — RLCD fine-tuning lifts it to 0.766. Conversely, smaller means better latency and footprint — a clean three-way quality/cost/speed trade-off.


## 3. Wrap-up — a map of the decision-model options

Including this folder (40_other_deciders), you have now experienced the same banking-support decisions five different ways.

| Approach | Decision maker | Decision latency (this env) | Cost structure | Notes |
|---|---|---|---|---|
| Bedrock Haiku 4.5 | General LLM generation | ~640ms | Token billing | Quality baseline, managed |
| SGLang /v1/systemone (01-05) | The Qwen3-VL-8B you already serve | **~35ms** | ~zero marginal cost | Multimodal, reuses existing serving |
| Bedrock Intelligent Routing (06) | Picks the **generation model** (not a decision) | Selected model's generation time | $1/1,000 req + tokens | Generation cost optimizer |
| strands-decider 2B (07) | Dedicated decision model | ~80ms | Self-hosted | Calibrated confidence, Strands SDK integration |
| Laya 322-421M (08) | Dedicated decision model | **~30ms** (in-process) | Self-hosted (CPU OK) | Multilingual auto-routing, smallest footprint |

- **The Korean angle**: Laya auto-selects its multilingual checkpoint and treats Korean as a first-class citizen — the dedicated model that fits this workshop's Korean scenarios most naturally.
- **Caution**: the meaning of "confidence" (how it is calibrated) differs per vendor. Thresholds do not transfer between products — always re-measure reliability on your own data.
- When zero-shot is not enough, Laya ships an RLCD fine-tuning path (official notebooks) — base 0.362 -> fine-tuned 0.766 on the typed-decisions benchmark.
